# Data Preperation

In [16]:
import xarray as xr
import matplotlib.pyplot as plt
import warnings
import numpy as np
import zarr
import pandas as pd
import warnings
#warnings.filterwarnings("ignore")
%matplotlib inline
###### opens the rave and weather data ######
file_path = "combined_final_backup.nc"
#ds = xr.open_dataset(file_path)
###### opens the stations data ######
pm2025path = "/home/exouser/code/station_datapm25/2025stationpm25hourly.csv"
df = pd.read_csv(pm2025path) 
station = df.to_xarray()

/tmp/ipykernel_87736/2355239997.py:15: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(pm2025path)


In [31]:
#zarr_path = "data\\20250107_1000-20250108_1000_WF.zarr"
#zarr_path = "/media/volume/firebackup/small_test/20250801_0000-20250803_2300_WF.zarr"
zarr_path = "/media/volume/firebackup/med_test/20250801_0000-20250831_2300_WF.zarr"

# 1. Dynamically find available fires to prevent ID KeyError
store = zarr.open_group(zarr_path, mode='r')
available_fires = list(store.group_keys())

if not available_fires:
    print("No fires found in the Zarr store!")
else:
    print(f"Fires available on disk: {available_fires}")
    
    # 2. Pick the first fire in the list (Change [0] to [1], etc. to view others)
    fire_id = available_fires[3] 

    # MUST use consolidated=False to read the appended stream correctly!
    ds = xr.open_zarr(zarr_path, group=fire_id , consolidated=False)  # group=fire_id 
    fire_name = ds.attrs.get('Fire_Name', '')

Fires available on disk: ['2025-CAFKU-018739_custom_cut', '2025-CASHF-001036_custom_cut', '2025-ORLAD-250494_custom_cut', '2025-CARRU-122253_custom_cut', '2025-CAMDF-000761_custom_cut', '2025-CALAC-283143_custom_cut', '2025-CANOD-004894_custom_cut', '2025-CASKU-006214_custom_cut', '2025-CATCU-012060_custom_cut', '2025-CAFKU-018813_custom_cut', '2025-CANEU-022955_custom_cut', '2025-NVEKD-010319_custom_cut', '2025-NVEKD-010346_custom_cut', '2025-CAAEU-025191_custom_cut', '2025-CAAEU-023830_custom_cut', '2025-CALAC-282478_custom_cut', '2025-CABDF-011350_custom_cut', '2025-NVTMFX-030516_custom_cut', '2025-NVEKD-010324_custom_cut', '2025-CALPF-002181_custom_cut', '2025-NVELD-040260_custom_cut', '2025-CAKNF-005712_custom_cut', '2025-NVEKD-010269_custom_cut', '2025-NVEKD-010270_custom_cut', '2025-CAMMU-020037_custom_cut', '2025-CANOD-005152_custom_cut', '2025-CAMMU-020033_custom_cut', '2025-CATCU-012044_custom_cut', '2025-CAMDF-000940_custom_cut', '2025-CAMMU-019994_custom_cut', '2025-CASQF-0

In [32]:
ds.load()

<xarray.Dataset> Size: 4GB
Dimensions:              (time: 744, y: 456, x: 415)
Coordinates:
  * time                 (time) datetime64[ns] 6kB 2025-08-01 ... 2025-08-31T...
    lat                  (y, x) float64 2MB 29.89 29.9 29.9 ... 44.39 44.4 44.4
    lon                  (y, x) float64 2MB 235.9 235.9 235.9 ... 245.9 245.9
    step                 timedelta64[ns] 8B 00:00:00
    heightAboveGround    float64 8B 10.0
    gribfile_projection  float64 8B nan
    valid_time           datetime64[ns] 8B 2025-08-31T23:00:00
    surface              float64 8B 0.0
Dimensions without coordinates: y, x
Data variables:
    d2m                  (time, y, x) float32 563MB 290.7 290.7 ... 273.2 273.3
    t2m                  (time, y, x) float32 563MB 292.0 292.0 ... 301.0 300.6
    sp                   (time, y, x) float32 563MB 1.02e+05 ... 8.259e+04
    rave_frp             (time, y, x) float32 563MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    u10                  (time, y, x) float32 563MB -1.624 -1.561 ... -2.426
    elevation            (time, y, x) float32 563MB 0.01339 0.01339 ... 1.79e+03
    v10                  (time, y, x) float32 563MB -7.289 -7.289 ... 1.687
Attributes: (12/13)
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    ...                      ...
    product:                 sfc
    description:             High-Resolution Rapid Refresh - CONUS
    remote_grib:             https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr....
    local_grib:              /media/volume/firebackup/med_test/raw_hrrr/hrrr/...
    search:                  :TMP:2 m|:DPT:2 m|:UGRD:10 m|:VGRD:10 m|:PBLH:|:...
    TEMPORAL_CLIP_STATUS:    FULLY_CONTAINED

med_test/20250801_0000-20250831_2300_WF.zarr

python collect_frp.py --zarrPath "/media/volume/firebackup/med_test/20250801_0000-20250831_2300_WF.zarr" --statPath "/home/exouser/code/station_datapm25/2025stationpm25hourly.csv" --distance "50000" --safe "False" --start "2025-08-01 20:00" --end "2025-08-03 12:00"

In [19]:
#assgining coordinates to the station data for more efficent indexing by time
date_hour = station['Date Local'] + ' ' + station['Time Local']
station = station.assign_coords({'lat':station['Latitude'],'long':station['Longitude'],'date_hour':date_hour,})

# Collective rave frp data creation

In [20]:
# start_date, end_date must have the given format:  'YYYY-MM-DD HH:MM'  and the space is needed too
# gives you the pandas date ranges for the given values in both the base, and extended forms (which we need later)
def make_date_ranges(start_date='2025-03-07 20:00',end_date='2025-03-07 20:00'):
    dates = pd.date_range(start=start_date, end=end_date,freq='h') #this is the date range we are interested in
    date_r = dates.strftime('%Y-%m-%d %H:%M')
    dates_str = dates.astype(str)
    dates_str = dates_str.str.replace(' ','T')
    dates_str =dates_str.str.cat([".000000000"]*len(dates_str))
    return date_r,dates_str

In [21]:
#we additionally prepare the dates to be the right format for indexing
date_r,dates_str = make_date_ranges('2025-08-01 20:00','2025-08-03 12:00')

In [22]:
from pyproj import CRS, Transformer#these are the transformers for projections 
crs_aea_epsg = CRS.from_epsg(5070) #albers eaqual area projection
transformer = Transformer.from_crs("EPSG:4326", crs_aea_epsg, always_xy=True) 

In [9]:
verboseee = False

In [29]:

############ get_collective_frp ############
# x_coord,y_coord: the projected x,y coordinate of the station
# x_relavent,y_relavent: the 2 LISTS, of projected x coords, and projected y coords for each fire
# clean_frp: the list of frp values, a list in parellel with x_relavent,y_relavent
# clean_u10,clean_v10: the 2 LISTS, of u and v components of wind vectors at the fires, a list in parellel with x_relavent,y_relavent
# safe: the safety device that stops the evaluation of this frp value if a invalid angle is found, turning this on should only be used for testing
############ Returns: ############
# the collective frp for the sation within the range
def get_collective_frp(x_relavent,y_relavent,x_coord,y_coord,clean_u10,clean_v10,clean_frp,safe=True):
    fire_wind = np.array([clean_u10,clean_v10]).T #TODO: CHECK IF THIS IS THE CORRECT ORDER
    fires_xy = np.array([x_relavent,y_relavent]) #array of fire positions (row 0 is the x coords, row 1 is the y coords)
    station_temp = np.tile([x_coord,y_coord], (fires_xy.shape[1], 1)).T
    
    fires_to_station = (station_temp - fires_xy).T#these are the station to fire vectors (tranpose for convenice later)
    #this order ensures that fire vec + distance vec points to the station(what we want)
    ############### Now we will calcuate the wind angle to direction to the station for each fire ###############
    wind_dist_dot = np.einsum('ij,ij->i', fires_to_station, fire_wind)#this calcuates the dot product for each pair of vectors across the two lists (ie for each fire)
    wind_norm = np.linalg.norm(fire_wind,axis=1) #norm of wind vectors 
    dist_norm = np.linalg.norm(fires_to_station,axis=1) #norm of the distance from fire to 
    wind_dist_prod = np.multiply(wind_norm,dist_norm) #element wise multiplication o fthe norms
    interal_value = np.divide(wind_dist_dot,wind_dist_prod) 
    #sanity check 
    if interal_value.max() > 1 or interal_value.min() < -1:
        #TODO: implelment some system such that this fire is ignored and the system can contine that is better then the current version
        #TODO: implelment some system such that this fire is ignored and the system can contine that is better then the current version
        if safe:
            raise ValueError(f"When calcuating the angle a value was found that would give a invalid angle when run through arccos, mainly a value greater then 1 or less then -1. min:{interal_value.min()}, max:{interal_value.max()}")
        else: #this is the unsafe option, where we will try and correct the angle. This is not recommended but is a work around for testing reasons
            toobig_vecs = np.where(interal_value > 1)
            toosmall_vecs = np.where(interal_value < -1)
            interal_value[toobig_vecs] = 1
            interal_value[toosmall_vecs] = -1
    #* there is a chance something in here is -1, or 1, if the vectors are in perfect alignment or agasint
    inline_vecs = np.where(interal_value == 1)
    opposite_vecs = np.where(interal_value == -1)
    angles = np.arccos(interal_value) 
    
    angles[inline_vecs] == 0
    angles[opposite_vecs] == 2*np.pi #manuely update those values to what they should be
    #^ BIG ASSUMPTION RIGHT HERE, WE ARE ASSUMING THAT THE LENGTH OF THE WIND VECTORS CORRESPONDS TO THE SPEED OF THE WIND ^#
    
    step1 = np.multiply(np.absolute(wind_norm),(np.cos(np.absolute(angles))+1)) #this is the forumla itself
    step2 = np.multiply(clean_frp,step1)
    step3 = 4*np.pi*((dist_norm/1000)**2)
    frps = np.divide(step2,step3)
    print(frps)
    collective_frp_value = frps.sum() #! RIGHT NOW THIS IS COMING OUT TO ZEROS BECAUSE THE SUBSECTION WE ARE WORKING WITH HAS NO FIRES
    return collective_frp_value

# Testing the get_collective_frp function:

Make the get_collective_frp function print the angles, and return list of frp values and you will see that this fits the interpretation of the graph i made a picture of on 3/27/26 around 11:30am MST, so go see that for reference for why this actually makes sense

In [10]:
x_relavent = np.array([3,1,3,4])
y_relavent = np.array([3,1,1,4])
x_coord = 2
y_coord = 2
clean_u10 = np.array([1,.5,0,-1])
clean_v10 = np.array([1,.5,1,-1])
clean_frp = np.array([4,4,4,4])
safe=True
get_collective_frp(x_relavent,y_relavent,x_coord,y_coord,clean_u10,clean_v10,clean_frp,safe)

np.float64(0.05626976975981911)

In [24]:
########### get_close_data ###########
# sta: the station itself we are using
# weather_frp_region: the weather and frp, we use this to get the cell closest to the station
# mask_lon: the lon values of the fire cells (these are still xarray datasets form )
# mask_lat: the lon values of the fire cells
# masked: the masked weather and frp data to being only fires
# dist_away: how far away fires can be to be considered
########### Returns: ###########
# fires_near: the dataset that was masked to be just the fires in range
# mask_x, mask_y: the transformed x,y coords of the fires
# dist_mask: the mask of fires that are within
# x_coord, y_coord: the x,y coords (projected) of the station
# station_weather_near: the weather cell nearest the station
def get_close_data(sta,weather_frp_region,mask_lon,mask_lat,masked,dist_away=10000000):
    ###### find the closest weather cell of the station
    ##### This uses base euclidean distance but because the distances will be rather small itll be fine i think #####
    #####TODO: change this to use the porjected coords you calcuated eariler :TODO#####
    #####TODO: change this to use the porjected coords you calcuated eariler :TODO#####
    dist = (weather_frp_region.lat - sta.lat.values)**2 + (weather_frp_region.lon - sta.long.values)**2
    min_idx = dist.argmin(dim=["y", "x"]) # Find the (y, x) index of the minimum distance 
    #this gets the weather info for the cell closest to the station
    station_weather_near = weather_frp_region.isel(y=min_idx["y"].values, x=min_idx["x"].values)
    
    ########## now we will find all the fires within the region of the station ##########
    ########## this uses the alber equal area projection formula ##########
    x_coord, y_coord = transformer.transform(sta.long.values, sta.lat.values) #this is the station lat longs now transformed into 2d coords 
    mask_x,mask_y = transformer.transform(mask_lon, mask_lat)
    x_diff = mask_x - x_coord #gets the difference between each point 
    y_diff = mask_y - y_coord
    diff = np.array([x_diff,y_diff])
    dist = np.linalg.norm(diff,axis=0) #now calcuates the distance between each point (now of shape 14x43 or the orginal shape of mask_lon and etc)
    dist_mask = dist < dist_away #makes a mask of all points that are within the needed distance dist_away
    dist_mask_np = xr.DataArray(dist_mask,dims=["y", "x"]) #makes the mask a xarray
    fires_near = masked.where(dist_mask_np) #!it seems this nans out more then i think #, drop=True
    
    return fires_near, mask_x,mask_y,dist_mask,x_coord, y_coord,station_weather_near

# NOTE, ablers equal area projection is in meters

In [30]:
safe=False
if safe == False:
        warnings.warn("This is the unsafe option where we will try and correct the angle when calcuating it, this is not recommended and should only be used for testing")
all_frps = []
all_ind = []
all_time = []
for date_hour,date_hour_extended in zip(date_r,dates_str): #for each timestamp
    stations_date_hour = station.sel(indexers={'date_hour':date_hour}) #gets the station info for that hour
    weather_frp_region = ds.sel(indexers={'time':date_hour_extended},method='nearest')
    masked = weather_frp_region.where(weather_frp_region.rave_frp > 0, drop=True) #288.4 #this nans out things that dont meet the conditions
    if np.isnan(masked.rave_frp.values).all(): #if no valid fires
        print('No fires for this time step, moving on')
        continue
    mask_lon = masked.lon #gets the all the lat lon values for the weather fire data
    mask_lat = masked.lat
    mask_frp = masked.rave_frp
    for i,thing in enumerate(stations_date_hour.index): #for each station by index(we can use the index to build other things)
        ind = thing.index.values #gets this stations index 
        sta = stations_date_hour.sel(indexers={'index':ind}) #get that specifc station by its index
        #NOTE, what we pass in: 'weather_frp_region' is masked such that it should only have valid fires, however with how the 'drop=True' functionality works, what it
        # drops is rows and columns of the dataset that are all naned out, HOWEVER, if a row or column still has regular values, but a nan or two in there, those nans will stay.
        # The net result is that while a var in 'weather_frp_region' may have dimension 11 x 10, it doesnt have 110 non nan values. This explains why you need two checks later on.
        # One is needed to ensure all values are within the distance, and whether they are valid fires.
        fires_near, mask_x,mask_y,dist_mask,x_coord, y_coord,station_weather_near = get_close_data(sta,weather_frp_region,mask_lon,mask_lat,masked,dist_away=50000)
        if dist_mask.sum() == 0: #then there is no fires within the distance and move on
             continue
        frp = fires_near.rave_frp.values.flatten() #now we pull out all the relavent data we need and clean it
        lat_rel = fires_near.lat.values.flatten()
        lon_rel = fires_near.lon.values.flatten()
        u10 = fires_near.u10.values.flatten()
        v10 = fires_near.v10.values.flatten()
        clean_frp = frp[~np.isnan(frp)] #this removes the nan values 
        clean_u10 = u10[~np.isnan(u10)]
        clean_v10 = v10[~np.isnan(v10)]
        lat_clean = lat_rel[~np.isnan(frp)]
        lon_clean = lon_rel[~np.isnan(frp)]
        x_relavent, y_relavent = transformer.transform(lon_clean, lat_clean)
        # Second sanity check: make sure that the fires_near outputed at least 1 valid fires
        if x_relavent.shape[0] == 0: 
            continue
        frp = get_collective_frp(x_relavent,y_relavent,x_coord,y_coord,clean_u10,clean_v10,clean_frp,safe)
        all_frps.append(frp)
        all_ind.append(ind.item())
        all_time.append(date_hour)
#station['collective_frp'] = ('index',all_frps)

/tmp/ipykernel_87736/4212641922.py:3: UserWarning: This is the unsafe option where we will try and correct the angle when calcuating it, this is not recommended and should only be used for testing
  warnings.warn("This is the unsafe option where we will try and correct the angle when calcuating it, this is not recommended and should only be used for testing")


[4.77914053e-06 9.68758163e-05 6.00434112e-06]
[0.00027333]
[7.86887438e-08 2.33316436e-07 4.50917970e-06 1.22444655e-03
 3.90711284e-04 1.05443277e-03 3.43418423e-04 5.54257270e-04
 2.89445193e-04 1.11769656e-05 1.22681751e-04 9.63667448e-05
 4.98529645e-06 5.28233568e-06 3.86966162e-04 7.45713248e-05
 3.57719389e-05 2.95120909e-06]
[0.00207023 0.01160786 0.00076373 0.00069111]
[0.00243688 0.01837074 0.00099913 0.00140214]
[7.14351378e-06 1.82109658e-04 1.28184019e-05]
[0.00044748]
[4.42531779e-09 3.96014877e-08 3.88216812e-06 1.13759097e-03
 1.66177766e-04 4.58510035e-04 2.39921638e-07 6.82055375e-04
 3.78125534e-04 1.10744339e-05 1.99120029e-04 1.46290268e-04
 8.09428161e-06 7.23725846e-06 6.85018663e-04 1.38115301e-04
 7.41211427e-05 7.55600512e-06]
[0.00288238 0.00248941 0.01567584 0.00116259 0.00059753 0.00108524]
[0.00321222 0.00393659 0.02353043 0.00154937 0.00108665 0.00199427]
[5.46710316e-06 1.51444398e-04 1.15467942e-05]
[8.10861511e-05]
[1.93120268e-06 3.58895751e-05 3.564

KeyboardInterrupt: 

In [28]:
np.array(all_frps).max()

np.float64(0.20023675329420207)

In [16]:
frp_temp = xr.Dataset(
    data_vars={
        "collective_frps": (["index"], np.array(all_frps)),          # variable C indexed by a shared dimension
    },
    coords={
        "index": ("index", all_ind),
        "date_hour": ("index", all_time),
    }
)

In [19]:
#frp_temp.sel(indexers={'index':93441})
combined = xr.merge([station, frp_temp])

/tmp/ipykernel_41240/2239238425.py:2: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'index' ('index',) The recommendation is to set join explicitly for this case.
  combined = xr.merge([station, frp_temp])
/tmp/ipykernel_41240/2239238425.py:2: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  combined = xr.merge([station, frp_temp])


In [ ]:
df_combined = combined.to_dataframe()
df_combined.to_csv('combined.csv')

In [22]:
np.unique(combined.collective_frps.values)

array([2.72541923e-07, 2.95524200e-07, 1.47651838e-06, 4.04495157e-05,
       6.28835377e-05, 6.83432094e-05, 7.28928894e-05, 7.40010193e-05,
       8.10861511e-05, 8.49288768e-05, 1.01783079e-04, 1.07659298e-04,
       1.24262670e-04, 1.68458295e-04, 2.02071574e-04, 2.09126252e-04,
       2.73327852e-04, 3.05784418e-04, 3.07076329e-04, 3.59876109e-04,
       4.03000846e-04, 4.11392674e-04, 4.47482056e-04, 5.13318362e-04,
       5.54384324e-04, 1.21347428e-03, 2.84390675e-03, 2.84402979e-03,
       4.10325317e-03, 4.60228640e-03, 4.86377448e-03, 6.96002234e-03,
       1.29870474e-02, 1.36980384e-02, 1.46539973e-02, 1.51329262e-02,
       1.58352094e-02, 1.60021890e-02, 1.63445598e-02, 1.65278155e-02,
       1.68645040e-02, 1.72648974e-02, 2.32088903e-02, 2.38929921e-02,
       2.39836028e-02, 2.45314134e-02, 2.57090873e-02, 3.46817763e-02,
       3.53095337e-02, 3.93768293e-02, 3.99878055e-02, 4.41537787e-02,
       5.39782458e-02, 5.51203960e-02, 1.25159980e-01, 1.31726332e-01,
      

In [35]:
frp_temp = xr.Dataset(all_frps, dims=['date_hour','index'])
frp_temp

TypeError: Dataset.__init__() got an unexpected keyword argument 'dims'

make it so that the station appends the date time and station index with the frp, that way it can attach easily to the station one

In [11]:
statpath = "/home/exouser/code/labfetch/station_with_frp.csv"
df = pd.read_csv(statpath) 
station = df.to_xarray()
station

/tmp/ipykernel_87736/2475006624.py:2: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(statpath)


<xarray.Dataset> Size: 223MB
Dimensions:              (index: 960700)
Coordinates:
  * index                (index) int64 8MB 0 1 2 3 ... 960697 960698 960699
Data variables: (12/28)
    State Code           (index) int64 8MB 2 2 2 2 2 2 2 ... 72 72 72 72 72 72
    County Code          (index) int64 8MB 110 110 110 110 110 ... 97 97 97 97
    Site Num             (index) int64 8MB 4 4 4 4 4 4 4 4 4 ... 7 7 7 7 7 7 7 7
    Parameter Code       (index) int64 8MB 88502 88502 88502 ... 88502 88502
    POC                  (index) int64 8MB 3 3 3 3 3 3 3 3 3 ... 1 1 1 1 1 1 1 1
    Latitude             (index) float64 8MB 58.39 58.39 58.39 ... 18.22 18.22
    ...                   ...
    County Name          (index) object 8MB 'Juneau ' 'Juneau ' ... 'Mayagnez'
    Date of Last Change  (index) object 8MB '2025-06-30' ... '2025-09-29'
    lat                  (index) float64 8MB 58.39 58.39 58.39 ... 18.22 18.22
    long                 (index) float64 8MB -134.6 -134.6 ... -67.14 -67.14
    date_hour            (index) object 8MB '2025-01-01 00:00' ... '2025-06-0...
    collective_frps      (index) float64 8MB nan nan nan nan ... nan nan nan nan

In [12]:
date_hour = station['Date Local'] + ' ' + station['Time Local']
station = station.assign_coords({'lat':station['Latitude'],'long':station['Longitude'],'date_hour':date_hour,})

In [15]:
np.unique(station.collective_frps.values)

array([2.72541923e-07, 2.95524200e-07, 1.47651838e-06, 4.04495157e-05,
       6.28835377e-05, 6.83432094e-05, 7.28928894e-05, 7.40010193e-05,
       8.10861511e-05, 8.49288768e-05, 1.01783079e-04, 1.07659298e-04,
       1.24262670e-04, 1.68458295e-04, 2.02071574e-04, 2.09126252e-04,
       2.73327852e-04, 3.05784418e-04, 3.07076329e-04, 3.59876109e-04,
       4.03000846e-04, 4.11392674e-04, 4.47482056e-04, 5.13318362e-04,
       5.54384324e-04, 1.21347428e-03, 2.84390675e-03, 2.84402979e-03,
       4.10325317e-03, 4.60228640e-03, 4.86377448e-03, 6.96002234e-03,
       1.29870474e-02, 1.36980384e-02, 1.46539973e-02, 1.51329262e-02,
       1.58352094e-02, 1.60021890e-02, 1.63445598e-02, 1.65278155e-02,
       1.68645040e-02, 1.72648974e-02, 2.32088903e-02, 2.38929921e-02,
       2.39836028e-02, 2.45314134e-02, 2.57090873e-02, 3.46817763e-02,
       3.53095337e-02, 3.93768293e-02, 3.99878055e-02, 4.41537787e-02,
       5.39782458e-02, 5.51203960e-02, 1.25159980e-01, 1.31726332e-01,
      

# Model iteration over for training

In [14]:
import torch

class mlp_1_baseline(torch.nn.Module):
    
    def __init__(self,input_size):
        """ Note that input size and latent size are arguments here """
        super().__init__()

        # Linear encoder 
        self.layer1 = torch.nn.Linear(input_size,20)
        self.layer2 = torch.nn.Linear(20,20)
        self.output = torch.nn.Sequential(torch.nn.Linear(20,1),torch.nn.Sigmoid())
        
    def forward(self,X):
        z = self.layer1(X)
        z = self.layer2(z)
        z = self.layer2(z)
        z = self.layer2(z)
        value = self.output(z) 
        return value  
   
class mlp_2_baseline(torch.nn.Module):
    def __init__(self,inputsize, depth, width):
        super().__init__()
        self.inp = torch.nn.Linear(inputsize, width)
        self.blocks = torch.nn.ModuleList([
            torch.nn.Sequential(
                torch.nn.Linear(width, width),
                torch.nn.tanh(),
            ) for _ in range(depth)
        ])
        self.out = torch.nn.Linear(width, 1)

    def forward(self, x):
        x = self.inp(x)
        for block in self.blocks:
            x = block(x)
        return self.out(x)

# Backup code:

In [ ]:
start_date='2025-01-07 17:00'
end_date='2025-01-07 23:00'
dates = pd.date_range(start=start_date, end=end_date,freq='h') #this is the date range we are interested in
dates_compare=dates[1:len(dates)]
dates_train=dates[0:len(dates)-1]

#makes the date ranges for compariasion
date_r_compare = dates_compare.strftime('%Y-%m-%d %H:%M')
dates_str_compare = dates_compare.astype(str)
dates_str_compare = dates_str_compare.str.replace(' ','T')
dates_str_compare =dates_str_compare.str.cat([".000000000"]*len(dates_str_compare))

#makes the date ranges for training
date_r_train = dates_train.strftime('%Y-%m-%d %H:%M')
dates_str_train = dates_train.astype(str)
dates_str_train = dates_str_train.str.replace(' ','T')
dates_str_train =dates_str_train.str.cat([".000000000"]*len(dates_str_train))

In [15]:
dist = (weather_frp_region.lat - sta.lat.values)**2 + (weather_frp_region.lon - sta.long.values)**2
# Find the (y, x) index of the minimum distance
min_idx = dist.argmin(dim=["y", "x"])
        #this gets the weather info for the cell closest to the station
station_weather_near = weather_frp_region.isel(y=min_idx["y"].values, x=min_idx["x"].values)
        ########## now we will find all the fires within the region of the station ##########
        ########## this uses the alber equal area projection formula ##########
x_coord, y_coord = transformer.transform(sta.long.values, sta.lat.values) #this is the station lat longs now transformed into 2d coords 
mask_x,mask_y = transformer.transform(mask_lon, mask_lat)
x_diff = mask_x - x_coord #gets the difference between each point 
y_diff = mask_y - y_coord
diff = np.array([x_diff,y_diff])
dist = np.linalg.norm(diff,axis=0) #now calcuates the distance between each point (now of shape 14x43 or the orginal shape of mask_lon and etc)
        
dist_mask = dist < 10000000 #makes a mask of all points that are within the needed distance
dist_mask_np = xr.DataArray(dist_mask,dims=["y", "x"]) #makes the mask a xarray
        
fires_near = masked.where(dist_mask_np, drop=True)#applies the above mask to the weather data already constrained by time to now be constrained by location as well

### This is the primary loop that will create the collective rave frp data

In [16]:
for date_hour,date_hour_extended in zip(date_r,dates_str): #for each timestamp
    stations_date_hour = station.sel(indexers={'date_hour':date_hour}) #gets the station info for that hour
    weather_frp_region = ds.sel(indexers={'time':date_hour_extended}) #gets the weather info for that hour
    ##### Currently this only nans out things unrelated to rave, but thats just because of the subset of the data im testing on #####
    ##### So later on i will change this to nan out things without a frp value
    masked = weather_frp_region.where((weather_frp_region.t2m > 288.6), drop=True) #288.4 #this nans out things that dont meet the conditions

    mask_lon = masked.lon#.values.flatten() #gets the all the lat lon values for the weather fire data
    mask_lat = masked.lat#.values.flatten()
    mask_frp = masked.rave_frp#.values.flatten()
    lon_lat_reference = np.array([mask_lon,mask_lat]).T #this is a list of lat longs that we will use for referenece later
    collective_frps = np.zeros(stations_date_hour.index.values.shape[0]) #this is where we will store each stations collective frp values (we will append to this as we go)

    for thing in stations_date_hour.index: #for each station by index(we can use the index to build other things)
        ind = thing.index.values #gets this stations index 
        sta = stations_date_hour.sel(indexers={'index':ind}) #get that specifc station by its index
        ##### Gets the distance of this station to all lat lon coords in the weather/frp grid #####
        ##### This uses base euclidean distance but because the distances will be rather small itll be fine i think #####
        #####TODO: change this to use the porjected coords you calcuated eariler :TODO#####
        #####TODO: change this to use the porjected coords you calcuated eariler :TODO#####
        dist = (weather_frp_region.lat - sta.lat.values)**2 + (weather_frp_region.lon - sta.long.values)**2
        # Find the (y, x) index of the minimum distance
        min_idx = dist.argmin(dim=["y", "x"])
        #this gets the weather info for the cell closest to the station
        station_weather_near = weather_frp_region.isel(y=min_idx["y"].values, x=min_idx["x"].values)
        ########## now we will find all the fires within the region of the station ##########
        ########## this uses the alber equal area projection formula ##########
        x_coord, y_coord = transformer.transform(sta.long.values, sta.lat.values) #this is the station lat longs now transformed into 2d coords 
        mask_x,mask_y = transformer.transform(mask_lon, mask_lat)
        x_diff = mask_x - x_coord #gets the difference between each point 
        y_diff = mask_y - y_coord
        diff = np.array([x_diff,y_diff])
        dist = np.linalg.norm(diff,axis=0) #now calcuates the distance between each point (now of shape 14x43 or the orginal shape of mask_lon and etc)
        
        dist_mask = dist < 10000000 #makes a mask of all points that are within the needed distance
        dist_mask_np = xr.DataArray(dist_mask,dims=["y", "x"]) #makes the mask a xarray
        
        fires_near = masked.where(dist_mask_np, drop=True)#applies the above mask to the weather data already constrained by time to now be constrained by location as well
    
        frp = fires_near.rave_frp.values.flatten() #now we pull out all the relavent data we need and clean it
        u10 = fires_near.u10.values.flatten()
        v10 = fires_near.v10.values.flatten()
        clean_frp = frp[~np.isnan(frp)] #this removes the nan values
        clean_u10 = u10[~np.isnan(u10)]
        clean_v10 = v10[~np.isnan(v10)]
        x_relavent = mask_x[dist_mask][~np.isnan(frp)] #the points of all things we want, now with the mask applied
        y_relavent = mask_y[dist_mask][~np.isnan(frp)] #while also removing the nan values
        
        #fires_to_station = np.array([x_relavent-x_coord,y_relavent-y_coord])
        fire_wind = np.array([clean_u10,clean_v10]).T #TODO: CHECK IF THIS IS THE CORRECT ORDER
        fires_xy = np.array([x_relavent,y_relavent])
        station_temp = np.tile([x_coord,y_coord], (2,int(.5*fires_xy.shape[1]))) #stacks the coords fires_xy.shape[1] times for vector subtraction
        ### btw the transposes is for easy of computation later
        fires_to_station = (station_temp - fires_xy).T#these are the station to fire vectors #?(NOTE: DOES THIS NEED TO BE MADE A ABSOLUTE VALUE, ASK JAKE)
        ############### Now we will calcuate the wind angle to direction to the station for each fire ###############
        wind_dist_dot = np.einsum('ij,ij->i', fires_to_station, fire_wind) 
        #this calcuates the dot product for each pair of vectors across the two lists (ie for each fire)
        wind_norm = np.linalg.norm(fire_wind,axis=1)
        dist_norm = np.linalg.norm(fires_to_station,axis=1)
        wind_dist_prod = np.multiply(wind_norm,dist_norm) #element wise multiplication
        interal_value = np.divide(wind_dist_dot,wind_dist_prod) #! which is caused by this step introducing something evil!!!!!
        angles = np.arccos(interal_value) #!in testing this is spiting out 1 NAN 
        #^ BIG ASSUMPTION RIGHT HERE, WE ARE ASSUMING THAT THE LENGTH OF THE WIND VECTORS CORRESPONDS TO THE SPEED OF THE WIND ^#
        step1 = np.multiply(wind_norm,np.cos(np.absolute(angles))) #this is the forumla itself
        print(angles)
        step2 = np.multiply(clean_frp,step1)
        step3 = 4*np.pi*((dist_norm)**2)
        #print(angles)
        frps = np.divide(step2,step3)
        collective_frp_value = frps.sum() #! RIGHT NOW THIS IS COMING OUT TO ZEROS BECAUSE THE SUBSECTION WE ARE WORKING WITH HAS NO FIRES
        




    

/tmp/ipykernel_5364/3336322692.py:62: RuntimeWarning: invalid value encountered in arccos
  angles = np.arccos(interal_value) #!in testing this is spiting out 1 NAN


[0.82451941 2.85285146 0.71985596 2.74651448 0.63997877 2.69842387
 0.59534408 2.64043423 0.52461102 2.58139753 0.48583449 3.02272617
 0.91166115 3.08079601 1.02885478 3.12854105 1.06445587 3.14074517
 0.99364694 3.04634611 0.88822576 2.95409261 0.81813342 2.8866775
 0.8008576  2.80171139 0.64489914 2.6616814  0.57225167 2.67101762
 0.61158456 2.67516881 0.54202899 2.55274658 0.91521811 3.05692661
 1.08521035 3.12705466 1.08641865 3.0998319  1.04307611 3.10204905
 0.94484916 3.01024183 0.87657687 2.85823926 0.80218759 2.83753654
 0.68975013 2.68592951 0.58072852 2.65204595 0.58148335 2.64661793
 0.54077002 2.55133947 0.4006491  2.81303357 1.08790061 3.10265462
 1.03768489 3.07567446 1.11778075 3.07474792 1.06072506 3.10890859
 0.94215538 2.99412803 0.81576497 2.89565928 0.79080397 2.79571053
 0.64576238 2.6513763  0.53157712 2.58431395 0.49639376 2.55162556
 0.44282636 2.478762   0.40116285 2.51020082 0.45215259 2.55968168
 0.53012373 2.72074882 1.49837241 3.04870152 1.00646074        

make testing, deal with nans from angles of vectors that are perfectly in line or oppeiste each other, also functionalis and make some fake data